# Portable Underwriter Report

This notebook uses one copied Python file and already-scored predictions. It does not need the model-build repository, SQL, Airflow, or a fitted model object. Put `portable_underwriter_report.py` beside this notebook before running it. The notebook environment needs NumPy, Pandas, Plotly, and PyArrow; the command-line form can provision them directly from the file's uv metadata.

## Input scales

Choose the model type explicitly because a library-neutral report cannot infer it from prediction columns.

| `model_type` | Actual and prediction | `sample_weight` |
|---|---|---|
| `frequency` | claim count / exposure | exposure |
| `severity` | claim cost / claim count | claim count |
| `burn_cost` | claim cost / exposure | exposure |

Pass final predictions on the response-rate scale, already including any model offset. An optional row-aligned `offset` column can be supplied for evidence-adapter binding; the report does not apply it again or serialize it.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from portable_underwriter_report import build_report

## Make a public synthetic scored sample

In real use, replace this cell with your scored DataFrame. Each prediction column must describe the same rows on the same response scale.

In [ ]:
rng = np.random.default_rng(24)
rows = 240
exposure = rng.uniform(0.4, 1.2, rows)
region = rng.choice(["North", "South", "West"], rows)
vehicle_age = rng.integers(0, 16, rows)
true_rate = 0.09 * np.exp(0.035 * vehicle_age + 0.18 * (region == "West"))
claim_count = rng.poisson(exposure * true_rate)

scored = pd.DataFrame(
    {
        "region": region,
        "vehicle_age": vehicle_age,
        "exposure": exposure,
        "actual_frequency": claim_count / exposure,
        "current_prediction": np.clip(0.105 * np.exp(0.018 * vehicle_age), 1e-6, None),
        "new_prediction": np.clip(
            0.09 * np.exp(0.032 * vehicle_age + 0.15 * (region == "West")),
            1e-6,
            None,
        ),
    }
)
scored.head()

## Build the HTML

The minimum cell size counts independent rows when `comparison_unit` is omitted. If several rows belong to one policy or customer, pass that identifier column as `comparison_unit`; its values are not written into the report.

In [ ]:
result = build_report(
    scored,
    actual="actual_frequency",
    predictions={
        "Current": "current_prediction",
        "New": "new_prediction",
    },
    sample_weight="exposure",
    features=["region", "vehicle_age"],
    # offset="report_time_offset",  # optional aligned adapter context
    model_type="frequency",
    output_path=Path("portable_model_review.html"),
    minimum_cell_size=20,
)
result.output_path

## Run the same thing from TOML

The copied file also works as a command:

```bash
uv run portable_underwriter_report.py --config report.toml
```

A prediction-only report always contains metrics, weighted prediction KDEs, model movement, Lorenz/gains, Gini, and double lift. Predictions alone cannot reconstruct fitted feature importance, isolated relativities, EDF, confidence intervals, or interactions. Those tabs accept optional library-neutral evidence objects when a model adapter can supply them; the base report never invents them.